# Notebook 02 — Pathway enrichment

For each selected component, run KEGG pathway enrichment over the PIP-significant features in each omic. RNA features are already gene symbols; methylation features are mapped to genes via the array manifest's `RefGene` column (first listed gene per CpG); proteins are stripped from `SYMBOL|UniProt` to `SYMBOL`.

**ChIP is not included in default enrichment.** The deposited H3K9ac peak manifest (`H3K9acDomains.csv`) carries chromosome coordinates and a Promoter/Enhancer/Other classification, but no gene-symbol or nearest-gene column. Mapping ChIP peaks to genes requires an external nearest-gene step (e.g., `bedtools closest`, or HOMER's `annotatePeaks.pl`), which is out of scope for these notebooks. The published manuscript's ChIP analysis is motif enrichment via HOMER (described in Methods), not pathway enrichment.

Returns a results dictionary `{component_id: {omic: DataFrame}}` you can flatten or save however you want.

## Required Synapse files

Download from the Synapse sub-project (accession TBD). Place all files in a
single directory and set `DEPOSIT_DIR` below.

| File | Shape | Description |
|---|---|---|
| `samples_final.csv` | 546 × 276 | Component × sample loadings |
| `cell_type.csv` | 546 × 6 | Component × cell-type loadings (Ast, Oli, Inh, OPC, Exc, Mic) |
| `RNA.txt` | 25,098 × 546 | Gene × component loadings (PIP > 0.5 masked) |
| `meth.txt` | 51,798 × 546 | CpG × component loadings (PIP > 0.5 masked) |
| `chip.txt` | 26,384 × 546 | H3K9ac peak × component loadings (PIP > 0.5 masked) |
| `proteins.txt` | 5,230 × 546 | Protein × component loadings (PIP > 0.5 masked) |

## Required reference annotations

Bundled in `references/` (or downloaded yourself — see references/README.md).

| File | Description |
|---|---|
| `Homo_sapiens.GRCh38.100.txt` | Gene annotation (Ensembl 100) |
| `ROSMAP_arrayMethylation_metaData.tsv` | Illumina methylation array manifest |
| `H3K9acDomains.csv` | H3K9ac ChIP-seq peak coordinates (Klein et al.) |

Clinical metadata (`ROSMAP_clinical.csv`) is **not redistributed** — it is
controlled-access (request via the [AD Knowledge Portal DUC]
(https://adknowledgeportal.synapse.org/Data%20Access)). Notebooks that need
clinical variables document their dependency inline and degrade gracefully
when the file is absent.


## Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the repo root to sys.path so we can import the helper module.
sys.path.insert(0, "..")
import g_ros_release as gr

gr.apply_plot_style()


In [2]:
# Path configuration — edit these two paths to match your local layout.
# `DEPOSIT_DIR` is where you downloaded the Synapse files (syn3219045 sub-
# project, accession TBD on publication). `REFERENCES_DIR` is the bundled
# `references/` directory in this repo.

# DEPOSIT_DIR is auto-resolved between two common layouts:
#   1. `synapse/` inside the cloned repo (default — what end-users get when
#      they download the deposit into a `synapse/` folder at the repo root)
#   2. `synapse/` as a sibling of the repo (the local-prep layout used to
#      keep code and data physically separate during preparation)
# To override either default, set the env var G_ROS_DEPOSIT_DIR.
import os as _os
DEPOSIT_DIR = _os.environ.get("G_ROS_DEPOSIT_DIR", "../synapse")
if not _os.path.exists(_os.path.join(DEPOSIT_DIR, "samples_final.csv")):
    _fallback = "../../synapse"
    if _os.path.exists(_os.path.join(_fallback, "samples_final.csv")):
        DEPOSIT_DIR = _fallback
        print(f"[notebook] using sibling deposit layout: {DEPOSIT_DIR}")
REFERENCES_DIR = "../references"             # bundled with this repo

deposit = gr.load_deposit(DEPOSIT_DIR, REFERENCES_DIR, load_references=True)
print(f"Loaded {len(deposit.samples)} components × {deposit.samples.shape[1]} samples")


Loaded 546 components × 276 samples


## Configuration

In [3]:
# Choose which components to run. Defaults to the six headline components
# featured in the manuscript. To sweep all 546 components, replace with:
#     COMPONENTS = list(range(1, 547))
# To run a single component, e.g. just Component 351:
#     COMPONENTS = [351]
#
# Headline components and their manuscript role (also in gr.HEADLINE_COMPONENTS):
#   351 — Figure 2 (microglial immunometabolic program)
#     2 — Figure 3 (multi-omic regulatory features)
#    74 — Figure 4 (sex-differentiated programs)
#    87 — Figure 5 (astrocyte immune-modulation)
#   278 — Sup Fig S4 (female-biased OPC/precursor shift)
#   177 — Sup Fig S5 (microglial bioenergetic program)

COMPONENTS = [351, 2, 74, 87, 278, 177]

# Omics to enrich. KEGG operates on gene symbols; the helper
# `gr.features_to_genes` maps each omic's features to gene symbols
# (RNA: identity; proteins: strip UniProt suffix; meth: nearest gene
# from the array manifest). ChIP is omitted by default — see notebook
# header for the reason.
OMICS = ['RNA', 'meth', 'proteins']

# gseapy enrichr cutoff (Adjusted P-value)
CUTOFF = 0.25

# Where to save enrichment tables. Set to None to skip saving.
SAVE_DIR = None  # e.g. "./out/kegg"
if SAVE_DIR is not None:
    os.makedirs(SAVE_DIR, exist_ok=True)


## Run enrichment

Each call to `gr.kegg_enrichment` makes a network request to the Enrichr API via gseapy. A full sweep of 546 components × 4 omics is ~2,184 requests — expect to be rate-limited and to take 30+ minutes. For the headline-component default (6 components), this runs in roughly a minute.

In [4]:
results: dict[int, dict[str, pd.DataFrame]] = {}

for cid in COMPONENTS:
    if not deposit.has_component(cid):
        print(f'Component {cid} not in deposit; skipping.')
        continue
    results[cid] = {}
    for omic in OMICS:
        gene_list = gr.features_to_genes(deposit, cid, omic)
        if not gene_list:
            print(f'  Component {cid} / {omic}: no PIP-significant features → skip')
            results[cid][omic] = pd.DataFrame()
            continue
        try:
            enr = gr.kegg_enrichment(deposit, cid, omic,
                                    gene_list=gene_list, cutoff=CUTOFF)
        except Exception as e:
            print(f'  Component {cid} / {omic}: enrichr call failed ({e}); skip')
            enr = pd.DataFrame()
        results[cid][omic] = enr
        n_terms = 0 if enr.empty else (enr['Adjusted P-value'] < CUTOFF).sum()
        print(f'  Component {cid} / {omic}: {len(gene_list)} genes → {n_terms} term(s) at adj.P < {CUTOFF}')

        if SAVE_DIR is not None and not enr.empty:
            out_path = os.path.join(SAVE_DIR, f'comp{cid}_{omic}_kegg.csv')
            enr.to_csv(out_path, index=False)


  Component 351 / RNA: 707 genes → 4 term(s) at adj.P < 0.25
  Component 351 / meth: 5 genes → 101 term(s) at adj.P < 0.25
  Component 351 / proteins: 315 genes → 51 term(s) at adj.P < 0.25
  Component 2 / RNA: 277 genes → 0 term(s) at adj.P < 0.25
  Component 2 / meth: 4 genes → 3 term(s) at adj.P < 0.25
  Component 2 / proteins: no PIP-significant features → skip
  Component 74 / RNA: 161 genes → 0 term(s) at adj.P < 0.25
  Component 74 / meth: enrichr call failed (Error sending gene list, status code: 429); skip
  Component 74 / meth: 1557 genes → 0 term(s) at adj.P < 0.25
  Component 74 / proteins: no PIP-significant features → skip
  Component 87 / RNA: 255 genes → 0 term(s) at adj.P < 0.25
  Component 87 / meth: 2 genes → 35 term(s) at adj.P < 0.25
  Component 87 / proteins: no PIP-significant features → skip
  Component 278 / RNA: 1114 genes → 177 term(s) at adj.P < 0.25
  Component 278 / meth: no PIP-significant features → skip
  Component 278 / proteins: no PIP-significant fea

## Inspect results — top terms per component, per omic

Display the top 5 enriched KEGG terms per (component, omic). Empty entries mean no term passed the cutoff.

In [5]:
for cid, by_omic in results.items():
    print(f'\n=== Component {cid} — {gr.HEADLINE_COMPONENTS.get(cid, "non-headline")} ===')
    for omic, df in by_omic.items():
        if df.empty:
            print(f'  [{omic}] no enriched terms')
            continue
        top = df.head(5)[['Term', 'Adjusted P-value', 'Combined Score']]
        print(f'  [{omic}] top {len(top)} of {len(df)}:')
        for _, row in top.iterrows():
            print(f"    {row['Term']}  (adj.P={row['Adjusted P-value']:.2e}, CS={row['Combined Score']:.1f})")



=== Component 351 — Figure 2 — microglial immunometabolic program ===
  [RNA] top 5 of 209:
    Cytokine-cytokine receptor interaction  (adj.P=3.07e-02, CS=27.5)
    Primary bile acid biosynthesis  (adj.P=7.41e-02, CS=81.5)
    Viral protein interaction with cytokine and cytokine receptor  (adj.P=7.41e-02, CS=33.3)
    Intestinal immune network for IgA production  (adj.P=1.29e-01, CS=35.9)
    Primary immunodeficiency  (adj.P=3.54e-01, CS=26.5)
  [meth] top 5 of 101:
    Endometrial cancer  (adj.P=6.10e-03, CS=2232.8)
    Pathways in cancer  (adj.P=6.10e-03, CS=477.2)
    Colorectal cancer  (adj.P=6.10e-03, CS=1361.5)
    Signaling pathways regulating pluripotency of stem cells  (adj.P=8.71e-03, CS=713.4)
    Breast cancer  (adj.P=8.71e-03, CS=688.6)
  [proteins] top 5 of 233:
    Inositol phosphate metabolism  (adj.P=1.97e-02, CS=59.5)
    mTOR signaling pathway  (adj.P=1.97e-02, CS=38.6)
    Shigellosis  (adj.P=4.33e-02, CS=24.7)
    Phosphatidylinositol signaling system  (adj.P=5.0

## Optional: flatten to a single long-format table

Useful for exporting a single CSV that anyone can filter, or for joining to `Table_S1.csv`.

In [6]:
rows = []
for cid, by_omic in results.items():
    for omic, df in by_omic.items():
        if df.empty:
            continue
        for _, r in df.iterrows():
            rows.append({
                'component_id': cid, 'omic': omic,
                'term': r['Term'],
                'adjusted_p': r['Adjusted P-value'],
                'combined_score': r['Combined Score'],
                'overlap': r.get('Overlap'),
                'genes': r.get('Genes'),
            })
long = pd.DataFrame(rows)
print(f'Long-format enrichment table: {long.shape}')
if not long.empty:
    display(long.head(20))


Long-format enrichment table: (1324, 7)


,component_id,omic,term,adjusted_p,combined_score,overlap,genes
0,351,RNA,Cytokine-cytokine receptor interaction,0.030664,27.491279,NaN,CCR1;IL10;TNFRSF13B;IL10RA;GDF6;INHBC;BMP6;CXC...
1,351,RNA,Primary bile acid biosynthesis,0.074079,81.498306,NaN,CH25H;ACOX2;CYP7A1;CYP8B1
2,351,RNA,Viral protein interaction with cytokine and cy...,0.074079,33.329392,NaN,CCR1;IL10;IL18RAP;IL10RA;IL2RA;CCR10;CXCL5
3,351,RNA,Intestinal immune network for IgA production,0.129182,35.915787,NaN,IL10;IL5;TNFRSF13B;CCR10;HLA-DRB1
4,351,RNA,Primary immunodeficiency,0.354121,26.458157,NaN,TNFRSF13B;CD8B;CD8B2;IL7R
5,351,RNA,Asthma,0.528895,27.090450,NaN,IL10;IL5;HLA-DRB1
6,351,RNA,Arginine biosynthesis,0.528895,24.629757,NaN,GOT1L1;ARG1;NOS3
7,351,RNA,Hematopoietic cell lineage,0.528895,11.597527,NaN,IL5;CD8B;IL2RA;CD8B2;IL7R;HLA-DRB1
8,351,RNA,Allograft rejection,0.673837,17.505439,NaN,IL10;IL5;HLA-DRB1
9,351,RNA,TGF-beta signaling pathway,0.757381,8.522404,NaN,BMP4;GDF6;THBS1;INHBC;DCN;BMP6
